In [3]:
!pip install torch torchvision torchaudio
!pip install transformers datasets scikit-learn
!pip install accelerate evaluate
!pip install nltk spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [4]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
# Import Libraries
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

import nltk
import spacy
import re
import string

In [6]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [7]:
# Check GPU + Mixed Precision Support
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Check AMP support
use_amp = torch.cuda.is_available()
print("Mixed Precision Enabled:", use_amp)

Using device: cuda
Mixed Precision Enabled: True


In [8]:
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu128
True


STEP 2: Efficient Dataset Loading (Streaming + Sampling)


In [9]:
#Loading dataset
from datasets import load_dataset

dataset = load_dataset(
    "dmitva/human_ai_generated_text",
    split="train",
    streaming=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

In [10]:
sample = next(iter(dataset))
print(sample)

{'id': 'cc902a20-27c4-4c18-8012-048a328206d1', 'human_text': "Also they feel more comfortable at home. Some school have decreased bullying and high and middle school because some students get bullied. Some Schools offter distance learning as an option for students to attend classes from home by way of online or video conferencing. Students can ncreased to learn at home. Also is more hard to students understand by online. students get distract at home. Some schools in United States ofter classes from home because is good option to students . Also students can stay more relaxing from home. Students don't want to go more at school and they want to get classes at home. Students get fall in environment learning. But students can get relaxing at home.\n\nStudents can get distract at home because they have easy to use phones. If students sleep in class they want to sleep at home too. They feel more bored at home because they need stay at home more time. Also students don't do anything at home

In [11]:
dataset = load_dataset(
    "dmitva/human_ai_generated_text",
    split="train",
    streaming=True
)

In [12]:
raw_samples = []

for i, item in enumerate(dataset):
    if i >= 5000:
        break
    raw_samples.append(item)

In [13]:
#reload dataset before using
dataset = load_dataset(
    "dmitva/human_ai_generated_text",
    split="train",
    streaming=True
)

In [16]:
from sklearn.model_selection import train_test_split

train_raw, val_raw = train_test_split(
    raw_samples,
    test_size=0.2,
    random_state=42
)

In [17]:
def expand_data(raw_data):
    texts = []
    labels = []

    for item in raw_data:
        if item.get("human_text"):
            texts.append(item["human_text"])
            labels.append(0)

        if item.get("ai_text"):
            texts.append(item["ai_text"])
            labels.append(1)

    return texts, labels

In [18]:
train_texts, train_labels = expand_data(train_raw)
val_texts, val_labels = expand_data(val_raw)

In [19]:
#Convert to Dataframe
import pandas as pd

train_df = pd.DataFrame({
    "text": train_texts,
    "label": train_labels
})

val_df = pd.DataFrame({
    "text": val_texts,
    "label": val_labels
})

In [21]:
train_df
val_df

,text,label
0,Self esteem come from praise or come from achi...,0
1,"Additionally, experiences of praise and accomp...",1
2,There are people who are sad and deperessed on...,0
3,But it can also limit students' flexibility in...,1
4,"For example, what if students don't attend the...",0
...,...,...
1995,"Similarly, Kobe Bryant is also renowned for co...",1
1996,"When we current know a person, our mind create...",0
1997,"In the case of the aforementioned restaurant, ...",1
1998,. The British naturalist and John Lub...,0


STEP 3: Stylometric Feature Engineering

In [22]:
import numpy as np
import nltk
import string
from collections import Counter

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [23]:
#Feature Function
# Type Token Ratio (TTR)
def type_token_ratio(text):
    tokens = nltk.word_tokenize(text)
    if len(tokens) == 0:
        return 0
    return len(set(tokens)) / len(tokens)

In [24]:
#Entropy
def entropy(text):
    tokens = nltk.word_tokenize(text)
    if len(tokens) == 0:
        return 0

    freq = Counter(tokens)
    probs = [f / len(tokens) for f in freq.values()]

    return -sum(p * np.log2(p) for p in probs)

In [25]:
#Avg Sentaence length
def avg_sentence_length(text):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) == 0:
        return 0

    words = [len(nltk.word_tokenize(s)) for s in sentences]
    return np.mean(words)

In [26]:
#Burstiness (variance of sentence length)
def burstiness(text):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= 1:
        return 0

    lengths = [len(nltk.word_tokenize(s)) for s in sentences]
    return np.var(lengths)

In [27]:
#Punctuation Density
def punctuation_density(text):
    if len(text) == 0:
        return 0
    punct_count = sum(1 for c in text if c in string.punctuation)
    return punct_count / len(text)

In [28]:
#Text Length
def text_length(text):
    return len(text)

In [105]:
#Combine Features into Vector
def extract_features(text):
    return [
        text_length(text),
        avg_sentence_length(text),
        type_token_ratio(text),
        entropy(text),
        punctuation_density(text),
        burstiness(text)
    ]

In [30]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [32]:
#Apply to Dataset
train_features = [extract_features(t) for t in train_df['text'][:2000]]

In [33]:
val_features = [extract_features(t) for t in val_df['text'][:1000]]

In [67]:
from sklearn.preprocessing import StandardScaler

scaler_feat = StandardScaler()

X_train_features = scaler_feat.fit_transform(X_train_features)
X_val_features = scaler_feat.transform(X_val_features)

In [68]:
import numpy as np

X_train_features = np.array(train_features)
X_val_features = np.array(val_features)

print(X_train_features.shape)
print(X_val_features.shape)

(2000, 6)
(1000, 6)


In [88]:
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler_feat = StandardScaler()

X_train_features = scaler_feat.fit_transform(X_train_features)
X_val_features = scaler_feat.transform(X_val_features)

# Extra safety (prevents extreme values → NaN)
X_train_features = np.clip(X_train_features, -5, 5)
X_val_features = np.clip(X_val_features, -5, 5)

In [89]:
y_train = train_df['label'][:2000].values
y_val = val_df['label'][:1000].values

STEP 4: Tokenization (DistilRoBERTa)

In [70]:
#Load Tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

In [71]:
#Define Tokenization Function
def tokenize_batch(texts, max_length=256):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

In [72]:
#Tokenize a Small Batch (Test First)
sample_texts = train_df['text'][:8].tolist()

encoded = tokenize_batch(sample_texts)

print(encoded.keys())

KeysView({'input_ids': tensor([[    0, 10787,  1294,  ..., 10010,    74,     2],
        [    0, 32591,     6,  ...,   521,     4,     2],
        [    0,  8275,    45,  ...,     1,     1,     1],
        ...,
        [    0,   100,   437,  ...,     1,     1,     1],
        [    0,   243,    18,  ...,  1980,    14,     2],
        [    0, 42713,     6,  ...,   115,   483,     2]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])})


In [73]:
#Inspect Shape
print(encoded['input_ids'].shape)

torch.Size([8, 256])


In [74]:
#Prepare Full Inputs (Aligned with Features)
train_texts_subset = train_df['text'][:2000].tolist()

train_encoded = tokenizer(
    train_texts_subset,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

train_input_ids = train_encoded['input_ids']
train_attention_mask = train_encoded['attention_mask']

In [75]:
val_texts_subset = val_df['text'][:1000].tolist()

val_encoded = tokenizer(
    val_texts_subset,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

val_input_ids = val_encoded['input_ids']
val_attention_mask = val_encoded['attention_mask']

In [44]:
#Labels
import torch

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

In [76]:
#Features
X_train_tensor = torch.tensor(X_train_features, dtype=torch.float32).to(device)
X_val_tensor = torch.tensor(X_val_features, dtype=torch.float32).to(device)

In [77]:
#Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_input_ids = train_input_ids.to(device)
train_attention_mask = train_attention_mask.to(device)
X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)

val_input_ids = val_input_ids.to(device)
val_attention_mask = val_attention_mask.to(device)
X_val_tensor = X_val_tensor.to(device)
y_val_tensor = y_val_tensor.to(device)

STEP 5: Build Hybrid Model (Transformer + Stylometric Features)

In [47]:
#Load Transformer Backbone
from transformers import AutoModel

transformer = AutoModel.from_pretrained("distilroberta-base")

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [119]:
#Create Hybrid Model Class
import torch.nn as nn
import torch

class HybridModel(nn.Module):
    def __init__(self, transformer, feature_dim=6):
        super(HybridModel, self).__init__()

        self.transformer = transformer
        hidden_size = transformer.config.hidden_size  # 768

        # Fusion Layer
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 1)  # Binary output
        )

    def forward(self, input_ids, attention_mask, features):
        # Transformer output
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # CLS token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (batch, 768)

        # Concatenate with features
        combined = torch.cat((cls_embedding, features * 0.05), dim=1)

        # Pass through classifier
        logits = self.classifier(combined)

        return logits

In [79]:
#Initialize Model
model = HybridModel(transformer)
model = model.to(device)

In [90]:
#Convert Features to Tensor
X_train_tensor = torch.tensor(X_train_features, dtype=torch.float32).to(device)
X_val_tensor = torch.tensor(X_val_features, dtype=torch.float32).to(device)

In [81]:
#Move Inputs to Device
# TRAIN
train_input_ids = train_input_ids.to(device)
train_attention_mask = train_attention_mask.to(device)
y_train_tensor = y_train_tensor.to(device)

# VALIDATION
val_input_ids = val_input_ids.to(device)
val_attention_mask = val_attention_mask.to(device)
y_val_tensor = y_val_tensor.to(device)

In [114]:
#Forward Pass Test
outputs = model(
    train_input_ids[:8],
    train_attention_mask[:8],
    X_train_tensor[:8]
)

print(outputs.shape)

torch.Size([8, 1])


STEP 6: Training Pipeline (AMP + Metrics + Early Stopping)

In [83]:
##Create DataLoader-like Batching
def create_batches(input_ids, attention_mask, features, labels, batch_size=16):
    for i in range(0, len(labels), batch_size):
        yield (
            input_ids[i:i+batch_size],
            attention_mask[i:i+batch_size],
            features[i:i+batch_size],
            labels[i:i+batch_size]
        )

In [120]:
#Loss + Optimizer + Scheduler
import torch.optim as optim
from transformers import get_linear_schedule_with_warmup

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(model.parameters(), lr=1e-5)

num_epochs = 5
batch_size = 16

total_steps = (len(y_train_tensor) // batch_size) * num_epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [85]:
#Mixed Precision Setup
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_5990/2844721782.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [86]:
# Evaluation Function
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def evaluate(model):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in create_batches(
            val_input_ids,
            val_attention_mask,
            X_val_tensor,
            y_val_tensor
        ):
            ids, mask, feats, labels = batch

            outputs = model(ids, mask, feats)
            probs = torch.sigmoid(outputs).cpu().numpy()

            all_preds.extend(probs)
            all_labels.extend(labels.cpu().numpy())

    preds_bin = [1 if p > 0.5 else 0 for p in all_preds]

    precision = precision_score(all_labels, preds_bin)
    recall = recall_score(all_labels, preds_bin)
    f1 = f1_score(all_labels, preds_bin)
    auc = roc_auc_score(all_labels, all_preds)

    return precision, recall, f1, auc

In [122]:
#Training Loop (with AMP)
best_f1 = 0
patience = 2
patience_counter = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in create_batches(
        train_input_ids,
        train_attention_mask,
        X_train_tensor,
        y_train_tensor
    ):
        ids, mask, feats, labels = batch

        optimizer.zero_grad()

        # Mixed Precision
        with torch.amp.autocast(device_type='cuda'):
            outputs = model(ids, mask, feats)
            loss = criterion(outputs.squeeze(), labels)

        # NaN safety check
        if torch.isnan(loss):
            print("Skipping NaN batch")
            continue

        # Backward
        scaler.scale(loss).backward()

        # Unscale before clipping (IMPORTANT)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Track optimizer step
        prev_scale = scaler.get_scale()

        scaler.step(optimizer)
        scaler.update()

        # Step scheduler ONLY if optimizer stepped
        if scaler.get_scale() <= prev_scale:
            scheduler.step()

        total_loss += loss.item()

    # VALIDATION
    precision, recall, f1, auc = evaluate(model)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"AUC: {auc:.4f}")

    # EARLY STOPPING
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered")
        break


Epoch 1
Loss: 4.5308
Precision: 0.9940
Recall: 1.0000
F1: 0.9970
AUC: 1.0000

Epoch 2
Loss: 1.9907
Precision: 0.9921
Recall: 1.0000
F1: 0.9960
AUC: 1.0000

Epoch 3
Loss: 1.2628
Precision: 0.9940
Recall: 1.0000
F1: 0.9970
AUC: 1.0000
Early stopping triggered


Inference Function (AI Probability Detector)

In [95]:
#Load Best Model
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

HybridModel(
  (transformer): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [97]:
#Reuse Feature Extractor
sample_text = "Artificial intelligence is transforming the world rapidly."

features = extract_features(sample_text)
print(features)

[58, np.float64(8.0), 1.0, np.float64(3.0), 0.017241379310344827, 0]


In [124]:
#Inference Function
def predict_ai_probability(text, temperature=3.0):
    model.eval()

    # Feature extraction
    features = extract_features(text)
    features = np.array(features).reshape(1, -1)
    features = scaler_feat.transform(features)
    features = np.clip(features, -5, 5)

    features_tensor = torch.tensor(features, dtype=torch.float32).to(device)

    # Tokenization
    encoded = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask, features_tensor)

        # Calibration
        logits = logits / temperature

        prob = torch.sigmoid(logits).item()

    return prob

In [116]:
#Labels
def classify_text(text, threshold=0.7):
    prob = predict_ai_probability(text)

    label = "AI-Generated" if prob > threshold else "Human-Written"

    return {
        "text": text,
        "AI_Probability": round(prob, 4),
        "Prediction": label
    }

In [125]:
tests = [
    "Hey dude what's up 😂",
    "bro I can't believe this happened lol",

    "This research paper explores the implications of artificial intelligence on society.",
    "Artificial intelligence has emerged as a transformative force in modern industries.",

    "idk man maybe later",
    "The methodology adopted in this study provides significant insights into neural architectures."
]

for t in tests:
    print(t, "→", predict_ai_probability(t))

Hey dude what's up 😂 → 0.20157316327095032
bro I can't believe this happened lol → 0.1423356682062149
This research paper explores the implications of artificial intelligence on society. → 0.8375940918922424
Artificial intelligence has emerged as a transformative force in modern industries. → 0.8378987312316895
idk man maybe later → 0.13314266502857208
The methodology adopted in this study provides significant insights into neural architectures. → 0.8376191258430481


In [126]:
#Test
print(classify_text("This essay explores the implications of machine learning..."))

{'text': 'This essay explores the implications of machine learning...', 'AI_Probability': 0.8361, 'Prediction': 'AI-Generated'}


In [127]:
predict_ai_probability("Artificial intelligence is transforming industries.")

0.8360249400138855

In [112]:
print(train_df['label'].value_counts())
print(val_df['label'].value_counts())


label
0    4000
1    4000
Name: count, dtype: int64
label
0    1000
1    1000
Name: count, dtype: int64


In [128]:
test_cases = [

    "Hey bro what's up 😂",
    "idk man maybe later, kinda busy rn",
    "lol that was insane dude",
    "ugh I forgot to submit the assignment again",

    "I think artificial intelligence is interesting but also confusing.",
    "This topic has both advantages and disadvantages.",

    "Artificial intelligence has emerged as a transformative force in modern society, influencing various domains.",
    "This research paper explores the implications of machine learning algorithms on data-driven decision making.",
    "The methodology adopted in this study provides significant insights into neural network optimization techniques.",
    "In recent years, technological advancements have significantly contributed to the evolution of intelligent systems."
]

for text in test_cases:
    prob = predict_ai_probability(text)
    print(f"{text} \n→ AI Probability: {prob:.4f}\n")

Hey bro what's up 😂 
→ AI Probability: 0.2637

idk man maybe later, kinda busy rn 
→ AI Probability: 0.1262

lol that was insane dude 
→ AI Probability: 0.1522

ugh I forgot to submit the assignment again 
→ AI Probability: 0.7177

I think artificial intelligence is interesting but also confusing. 
→ AI Probability: 0.7959

This topic has both advantages and disadvantages. 
→ AI Probability: 0.8361

Artificial intelligence has emerged as a transformative force in modern society, influencing various domains. 
→ AI Probability: 0.8382

This research paper explores the implications of machine learning algorithms on data-driven decision making. 
→ AI Probability: 0.8377

The methodology adopted in this study provides significant insights into neural network optimization techniques. 
→ AI Probability: 0.8378

In recent years, technological advancements have significantly contributed to the evolution of intelligent systems. 
→ AI Probability: 0.8384

